# SaúdeJá - Classificador de No-show v0.5 (Camila)

Objetivo: prever se um paciente vai faltar à consulta agendada.

Dataset: `data/consultas-historicas.csv` (sintético, ~380 consultas).

Stack: pandas + scikit-learn + LightGBM.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import lightgbm as lgb

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
df = pd.read_csv('data/consultas-historicas.csv')
print(df.shape)
df.head()

## EDA

Dataset pequeno (~400 linhas). Olhar rápido na distribuição.

In [ ]:
print(df.describe())
print('\n--- target ---')
print(df['no_show'].value_counts(normalize=True))
print('\n--- especialidade ---')
print(df['especialidade'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df['idade'].hist(bins=20, ax=axes[0]); axes[0].set_title('idade')
df['distancia_km'].hist(bins=20, ax=axes[1]); axes[1].set_title('distancia_km')
df['historico_noshow'].hist(bins=10, ax=axes[2]); axes[2].set_title('historico_noshow')
plt.tight_layout()
plt.show()

## Pré-processamento

sexo -> 0/1. especialidade -> label encoding (LightGBM lida bem).

In [ ]:
df['sexo'] = df['sexo'].map({'F': 0, 'M': 1})

especialidades = sorted(df['especialidade'].unique())
mapa_esp = {e: i for i, e in enumerate(especialidades)}
df['especialidade'] = df['especialidade'].map(mapa_esp)
print('mapa_especialidade:', mapa_esp)

features = ['idade', 'sexo', 'especialidade', 'distancia_km',
            'dias_entre_agendamento_consulta', 'historico_noshow']
X = df[features]
y = df['no_show']
print('X:', X.shape, 'y:', y.shape)

## Treino

Split 80/20 estratificado. LightGBM com hiperparâmetros tunados na mão.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print('Acurácia:', model.score(X_test, y_test))
print('ROC-AUC :', roc_auc_score(y_test, y_proba))
print('\n', classification_report(y_test, y_pred, digits=3))
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))

## Salvar modelo

In [ ]:
joblib.dump({'model': model, 'mapa_especialidade': mapa_esp}, 'model.pkl')
print('modelo salvo em model.pkl')

## Próximos passos (Camila)

- [ ] Transformar isso em API (FastAPI?) — ticket SAUDEJA-241
- [ ] SHAP / feature importance pra liderança médica
- [ ] SMOTE ou class_weight='balanced' pra subir F1 da classe positiva
- [ ] Calibrar threshold com base no custo do SMS vs. perda por no-show
- [ ] Adicionar features de dia-da-semana e horário da consulta
- [ ] Pipeline de re-treino mensal (cron)
- [ ] Testes unitários
- [ ] LGPD: revisar com DPO antes de qualquer dado real